<a href="https://colab.research.google.com/github/Raquelsantosb/analise-dados-python/blob/main/atividade01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

# 1. Carregar uma série temporal clássica da internet (AirPassengers)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
df = pd.read_csv(url, parse_dates=['Month'], index_col='Month')
df.columns = ['Passageiros']

# Criar variável de tempo determinística (t = 1, 2, 3...)
df['t'] = np.arange(1, len(df) + 1)

# Criar variáveis dummy para a sazonalidade (mensal)
# drop_first=True evita a armadilha da multicolinearidade perfeita
dummies = pd.get_dummies(df.index.month, prefix='mes', drop_first=True).set_index(df.index)
df = pd.concat([df, dummies], axis=1)


# ABORDAGEM 1: MODELO ADITIVO

# Y = beta0 + beta1*t + dummy_meses
X_add = sm.add_constant(df[['t']].join(dummies))
modelo_add = sm.OLS(df['Passageiros'], X_add).fit()

# Correção: Calcular a tendência isolada usando os coeficientes (constante + beta_1 * t)
df['tendencia_add'] = modelo_add.params['const'] + modelo_add.params['t'] * df['t']
df['ajustado_add'] = modelo_add.fittedvalues
df['residuo_add'] = df['Passageiros'] - df['ajustado_add']


# ABORDAGEM 2: MODELO MULTIPLICATIVO (Linearizado via Logaritmo)
# ln(Y) = beta0 + beta1*t + dummy_meses

df['ln_y'] = np.log(df['Passageiros'])
modelo_mult = sm.OLS(df['ln_y'], X_add).fit()

# Correção: Calcular a tendência na escala log e depois aplicar a exponencial
ln_tendencia_mult = modelo_mult.params['const'] + modelo_mult.params['t'] * df['t']
df['tendencia_mult'] = np.exp(ln_tendencia_mult)
df['ajustado_mult'] = np.exp(modelo_mult.fittedvalues)
df['residuo_mult'] = df['Passageiros'] - df['ajustado_mult']


# COMPARAÇÃO DE MÉTRICAS (R² e AIC)

print(f"R² Aditivo: {modelo_add.rsquared:.4f}")
print(f"R² Multiplicativo (na escala log): {modelo_mult.rsquared:.4f}")

R² Aditivo: 0.9559
R² Multiplicativo (na escala log): 0.9835
